# Week 1 — 데이터 탐색 + Classical Baseline (1조 Advanced)

**구 W1 + W2 통합판** (2026-06-05). 한 세션에 데이터 처리 → sparse 문제 정의 → classical baseline까지.

## 이번 주 학습 목표
1. **데이터 I/O + 전처리** — `.bin` 로드, float 정규화, Otsu 임계값 자동 선택
2. **3D voxel 데이터 시각화** — 4개 도메인(BB·CastleGate·Bentheimer·Parker) 비교
3. **공극률 분석 + 등방성 검증** — slab별 분해, 세 축 프로파일
4. **Sparse imaging 문제 정의** + **Classical baseline 정량 비교**
   - B1 (Linear) vs B2 (Cubic spline, scipy)
   - 평가 지표 다중화: `|Δφ|`, `|ΔSA|`, `SSIM`
5. **k sweep으로 "왜 deep learning이 필요한가" 정량 확인**

## 학습 방식

본 노트북은 **"배포된 코드를 읽고 / 파라미터를 바꿔보며 / 결과를 본인의 분석으로 해석한다"** 는 원칙으로 구성되어 있습니다. 직접 함수를 다시 짜기보다 "이 함수가 왜 이렇게 동작하는가"를 데이터로 검증하는 데 집중하세요.

본문에 **[탐구]** 블록이 있습니다 — 변수 sweep, 가설 수립·검증, 결과 비교를 자유롭게 시도. 정답이 정해진 "한 줄 답" 박스는 두지 않습니다. 본인 노트(별도 `.md` / 노트북 텍스트 셀)에 분석·의문·다음 가설을 자유롭게 남기는 형태가 권장됩니다.

마지막 **탐구 과제** 는 본 노트북을 fork하여 코드 수정 + 결과 시각화 + 본인 분석을 함께 정리해 제출.

## 0. 환경 준비 + helper 함수 import

In [ ]:
import sys

from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'helpers'))



import numpy as np

import matplotlib.pyplot as plt



from dr_utils import (

    # 데이터 I/O + 전처리

    load_volume, porosity, normalize_to_float, otsu_threshold, binarize_otsu,

    # 시각화

    show_three_axis, porosity_profile,

    # Sparse + baseline

    make_sparse, linear_interpolate_slice,

    reconstruct_sparse_linear, reconstruct_sparse_cubic,

    # 평가

    porosity_error, surface_area_error, ssim_3d_mean, summarize_metrics,

    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,

)

setup_plot_style()

print('환경 준비 완료')

## 1. 데이터 로드 — 4 도메인 동시 비교



| 도메인 | 출처 | 특징 |

|---|---|---|

| BB | Brazil sandstone | 기준 학습 도메인 |

| CastleGate | CastleGate sandstone | 고공극·불균질 |

| Bentheimer | Bentheimer sandstone | 균질 사암 |

| **Parker** | Parker sandstone | 저공극 (W1 신규 비교) |



모두 256³ uint8 binary (0=solid, 1=pore), voxel 2.25 μm.

In [ ]:
DATA_DIR = Path('..') / 'data'

domains = {

    'BB':         load_volume(DATA_DIR / 'BB_256.bin'),

    'CastleGate': load_volume(DATA_DIR / 'CastleGate_256.bin'),

    'Bentheimer': load_volume(DATA_DIR / 'Bentheimer_256.bin'),

    'Parker':     load_volume(DATA_DIR / 'Parker_256.bin'),

}



print(f"{'Domain':<13} {'shape':<18} {'dtype':<8} {'φ (%)':>8}")

print('-' * 52)

for name, vol in domains.items():

    print(f"{name:<13} {str(vol.shape):<18} {str(vol.dtype):<8} {porosity(vol)*100:>7.2f}")

## 2. 데이터 전처리 — float 정규화 + Otsu 임계값



Deep learning 모델 학습 전 표준 전처리. W2 이후 UNet 입력 단계에서 매번 필요.



**`normalize_to_float`**: uint8 [0, 255] → float32 [0.0, 1.0]

**`otsu_threshold`**: grayscale에서 자동 임계값 (binary 데이터에선 0.5 — trivial)

In [ ]:
vol = domains['BB']

print(f'원본 dtype = {vol.dtype}, range = [{vol.min()}, {vol.max()}]')



vol_f = normalize_to_float(vol)

print(f'정규화 후 dtype = {vol_f.dtype}, range = [{vol_f.min()}, {vol_f.max()}]')



# Otsu — 본 binary 데이터에서는 trivial (값이 0 또는 1뿐)

t = otsu_threshold(vol_f[128])

print(f'Otsu threshold (z=128 slice) = {t:.4f}  (binary 데이터 → trivial)')

> **[Try-it! ①]** Otsu는 grayscale 데이터에서 진가를 발휘. 인공 grayscale을 만들어 시각해봅시다.

In [ ]:
# 인공 grayscale: binary에 가우시안 노이즈 추가

rng = np.random.default_rng(0)

gray = vol[128].astype(np.float32) + rng.normal(0, 0.2, vol[128].shape)

gray = np.clip(gray, 0, 1)



t_gray = otsu_threshold(gray)

binary, _ = binarize_otsu(gray)



fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(gray); axes[0].set_title('인공 grayscale (noisy)')

axes[0].axis('off')

axes[1].hist(gray.ravel(), bins=80, color=GRAY)

axes[1].axvline(t_gray, color=ORANGE, lw=2, label=f'Otsu t={t_gray:.3f}')

axes[1].set_title('히스토그램 + Otsu'); axes[1].legend()

axes[2].imshow(binary); axes[2].set_title(f'Otsu binarize (φ={binary.mean()*100:.1f}%)')

axes[2].axis('off')

plt.tight_layout(); plt.show()

> **[해석 질문 1]** Otsu가 자동으로 고른 임계값이 "공극과 암석의 경계"에 잘 맞나요?

> 만약 노이즈 크기(`rng.normal(0, 0.2, ...)` 의 `0.2`)를 0.05 또는 0.5로 바꾸면 Otsu 결과가 어떻게 변할까요? 직접 sweep해보세요.

## 3. 4 도메인 시각화 + 등방성 분석

In [ ]:
# 4 도메인 중앙 슬라이스 (z=128) 한 줄에

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (name, vol) in zip(axes, domains.items()):

    ax.imshow(vol[128])

    ax.set_title(f'{name}  (φ={porosity(vol)*100:.1f}%)')

    ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
# BB의 세 축 프로파일 — 등방성 검증

vol = domains['BB']

n_slabs = 16

prof_z = porosity_profile(vol, axis=0, n_slabs=n_slabs)

prof_y = porosity_profile(vol, axis=1, n_slabs=n_slabs)

prof_x = porosity_profile(vol, axis=2, n_slabs=n_slabs)



fig, ax = plt.subplots(figsize=(9, 4))

xa = np.arange(n_slabs)

ax.plot(xa, prof_z, marker='o', label=f'z-axis (std={prof_z.std():.4f})', color=ORANGE, lw=2)

ax.plot(xa, prof_y, marker='s', label=f'y-axis (std={prof_y.std():.4f})', color=NAVY, lw=2)

ax.plot(xa, prof_x, marker='^', label=f'x-axis (std={prof_x.std():.4f})', color=GREEN, lw=2)

ax.axhline(porosity(vol), ls='--', color=GRAY, label=f'overall φ={porosity(vol):.3f}')

ax.set_xlabel(f'Slab index (n_slabs={n_slabs})'); ax.set_ylabel('φ')

ax.set_title('BB sandstone — 세 축 등방성 검증')

ax.legend(); plt.tight_layout(); plt.show()

> **[Try-it! ②]** 4 도메인 각각에 대해 `prof_z.std()` 를 계산해 등방성 순위를 매겨보세요.

> **본 연구의 tri-axis aggregation** (W5) 은 "세 축이 비슷한 통계" 라는 가정에 의존합니다 — 어느 도메인이 그 가정을 가장 잘 만족하나요?

## 4. Sparse imaging 문제 정의



**문제**: micro-CT 스캔은 시간/비용이 비쌈. k 슬라이스마다 1개만 측정하면 시간 (1−1/k)×100% 절감.

**핵심 질문**: 누락 슬라이스를 측정 슬라이스로부터 얼마나 정확히 복원할 수 있나?

In [ ]:
vol = domains['BB']

k = 5

known_idx, missing_idx = make_sparse(vol, k=k, axis=0)

print(f'k={k}: 측정 {len(known_idx)}장 / 누락 {len(missing_idx)}장 / 시간 절감 {(1-1/k)*100:.1f}%')



fig, axes = plt.subplots(1, 6, figsize=(14, 3))

for i, z in enumerate(range(60, 66)):

    axes[i].imshow(vol[z])

    is_known = z in known_idx

    axes[i].set_title(f'z={z}\n{"measured" if is_known else "MISSING"}',

                      color=GREEN if is_known else RED, fontsize=10)

    axes[i].axis('off')

plt.suptitle(f'Sparse k={k}', y=1.05); plt.tight_layout(); plt.show()

## 5. Classical Baseline — Linear (B1) vs Cubic (B2)



두 가지 고전 보간법으로 누락 슬라이스를 복원하고 정량 비교.



- **B1 (Linear)**: 인접 2 슬라이스 사이를 1차 직선으로 보간

- **B2 (Cubic, scipy)**: 4 known 슬라이스를 사용한 3차 spline (부드러움)



평가 지표 3종:

- `|Δφ|` (porosity error, %p) — 공극률 보존

- `|ΔSA|` (surface area error, /Mvoxel) — 표면적 보존

- `SSIM` (0~1, 1이 완벽) — 구조 유사도

In [ ]:
vol = domains['BB']

k = 5



print(f'BB sandstone, k={k} (시간 80% 절감)')

print('-' * 65)

recon_l = reconstruct_sparse_linear(vol, k=k)

m_l = summarize_metrics(recon_l, vol, label='B1 Linear')

recon_c = reconstruct_sparse_cubic(vol, k=k)

m_c = summarize_metrics(recon_c, vol, label='B2 Cubic')

In [ ]:
# 시각: 원본 vs B1 vs B2 + 오차맵

z_show = 62  # missing slice (62 % 5 != 0)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(vol[z_show]); axes[0].set_title(f'원본 z={z_show}')

axes[1].imshow(recon_l[z_show]); axes[1].set_title(f'B1 Linear')

axes[2].imshow(recon_c[z_show]); axes[2].set_title(f'B2 Cubic')

diff_l = np.abs(vol[z_show].astype(float) - recon_l[z_show])

axes[3].imshow(diff_l, cmap='hot'); axes[3].set_title(f'|원본 − B1|')

for ax in axes: ax.axis('off')

plt.tight_layout(); plt.show()

> **[Try-it! ③]** `k` 를 3, 5, 7, 10 으로 바꾸며 B1·B2 의 세 지표 변화를 표로 정리.

> 어느 k부터 cubic이 linear보다 의미 있게 좋은가요? (또는 그 반대?)

## 6. 4 도메인 × k sweep — 본 연구의 motivation 그래프



이 곡선이 "왜 deep learning이 필요한가" 의 정량 증거입니다.

W3에서 UNet 결과를 같은 plot에 겹쳐 비교할 것입니다.

In [ ]:
k_list = [2, 3, 5, 7]

results = {name: {'k': [], 'dphi': [], 'ssim': []} for name in domains}



for name, vol in domains.items():

    for k in k_list:

        rec = reconstruct_sparse_linear(vol, k=k)

        results[name]['k'].append(k)

        results[name]['dphi'].append(porosity_error(rec, vol) * 100)

        results[name]['ssim'].append(ssim_3d_mean(rec, vol))

    print(f'  {name:<12}  done')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {'BB': ORANGE, 'CastleGate': NAVY, 'Bentheimer': GREEN, 'Parker': RED}

for name, r in results.items():

    axes[0].plot(r['k'], r['dphi'], marker='o', lw=2, label=name, color=colors[name])

    axes[1].plot(r['k'], r['ssim'], marker='s', lw=2, label=name, color=colors[name])

axes[0].set_xlabel('k'); axes[0].set_ylabel('|Δφ| (%p)')

axes[0].set_title('Linear baseline — 공극률 오차')

axes[1].set_xlabel('k'); axes[1].set_ylabel('SSIM')

axes[1].set_title('Linear baseline — 구조 유사도')

for ax in axes: ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

> **[해석 질문 2]** 4 도메인 중 어느 사암이 "sparse 보간이 가장 어려운가"?

> 그 이유로 어떤 가설을 세울 수 있을까요? (힌트: 공극률, 등방성, 구조 복잡도)

## 7. 다음 주 (W2)

- 본 W1 baseline 곡선을 "deep learning이 어디까지 끌어내릴 수 있나" 직접 확인
- mini UNet (~100K params) 학생 노트북에서 학습 — 10/30/60분 옵션
- `pip install torch torchvision`

본인이 본 W1에서 정량적으로 확인한 baseline 한계 — 어떤 부분에서 가장 답답했는가? 그 답답함이 W2에서 deep learning을 만나는 출발점이 됩니다.

---

## 🎯 W1 탐구 과제 (1조 Advanced)

본 노트북을 본인 작업 파일로 복사한 뒤, 다음 과제를 본인 분석·시각화·해석과 함께 정리해 제출하세요.

### 과제 1 — 전수 baseline 비교 (필수)

4 도메인 × {B1 Linear, B2 Cubic} × k ∈ {2, 3, 5, 7} 의 모든 조합에 대해 (|Δφ|, |ΔSA|, SSIM) 측정 → pandas DataFrame으로 정리 → 의미 있는 pivot/aggregation으로 해석.

핵심 질문 (참고용, 본인이 더 흥미로운 질문을 추가해도 좋습니다):
- 모든 도메인에서 B2가 B1보다 명확히 우수한가? 반례가 있다면 그 도메인의 특징은?
- k가 커질수록 어느 baseline이 더 빠르게 악화되는가?

### 과제 2 — 세 축 sparse 보간 (필수)

`make_sparse` / `reconstruct_sparse_*` 의 `axis` 인자를 0/1/2로 바꿔서 sparse 시뮬레이션이 어느 방향에서든 가능합니다. 본 데이터의 등방성이 정말 성립하는지, 세 축 보간 결과로 정량 검증.

이 결과가 본 연구에서 "세 축을 모두 활용" 하는 접근이 합리적인 이유와 어떻게 연결되는지 본인의 해석을 추가.

### 과제 3 — Otsu sensitivity (선택)

`reconstruct_sparse_*` 안의 이진화 임계값(`> 0.5`)이나, `binarize_otsu`의 동작 자체를 탐색. 인공 grayscale 데이터에 noise σ 를 다양하게 주면서 Otsu가 어디서 불안정해지는지 본인 실험 + 시각화.

### 과제 4 — Random sparse 시나리오 (선택, 도전)

본 노트북은 결정적 k 시나리오만 다룹니다. random sparse (예: 무작위 33% 측정)를 본인이 구현하여 deterministic k=3과 비교. 두 시나리오의 정보량은 같지만 보간 난이도는 다를 수 있습니다 — 그 이유를 본인 가설로 정리.